## Imports

In [1]:
import csv
import cv2
import json
import numpy as np
import pytesseract
import time
import os
from dotenv import load_dotenv
from pathlib import Path
from firebase_admin import credentials
from firebase_admin import firestore
import queue
import threading
import re
from datetime import datetime
from typing import List
import requests

from IPython.display import display, clear_output
from PIL import Image, ImageDraw

## Data Loads

In [2]:
# Load track translations (ocr output to correct track name)
with open("track_translations.json", "r") as f:
    TRACK_TRANSLATIONS = json.load(f)

# Load lap masks for players 1-4
P11 = np.load("player_1_lap_1.npy")
P12 = np.load("player_1_lap_2.npy")
P13 = np.load("player_1_lap_3.npy")
P21 = np.load("player_2_lap_1.npy")
P22 = np.load("player_2_lap_2.npy")
P23 = np.load("player_2_lap_3.npy")
P31 = np.load("player_3_lap_1.npy")
P32 = np.load("player_3_lap_2.npy")
P33 = np.load("player_3_lap_3.npy")
P41 = np.load("player_4_lap_1.npy")
P42 = np.load("player_4_lap_2.npy")
P43 = np.load("player_4_lap_3.npy")

LAP_MASKS = [
    [P11, P12, P13],
    [P21, P22, P23],
    [P31, P32, P33],
    [P41, P42, P43]
]

# Load finish text mask for players 1-4
P1FINISH = np.load("player_1_finish_gray.npy")
P2FINISH = np.load("player_2_finish_gray.npy")
P3FINISH = np.load("player_3_finish_gray.npy")
P4FINISH = np.load("player_4_finish_gray.npy")

FINISH_MASKS = [P1FINISH, P2FINISH, P3FINISH, P4FINISH]

# Load grayscale mask of the "Go!" icon
GO_MASK = np.load("go_gray.npy")

# Load the key for the Yalies API
YALIES_KEY = os.environ.get("YALIES_KEY")

## Hardcoded Values

In [3]:
LAP_REGIONS = [
    (152, 315, 15, 22),
    (1173, 315, 15, 22),
    (152, 660, 15, 22),
    (1173, 660, 15, 22)
]

FINISH_REGIONS = [
    (172, 128, 324, 72),
    (787, 128, 324, 72),
    (172, 474, 324, 72),
    (787, 474, 324, 72)
]

GO_REGION = (480, 240, 320, 144)

LOADING_REGION = (50, 50, 1180, 200)

TRACK_NAME_REGION = (261, 605, 800, 58)

RESULT_REGION_Y_VALUES = [67, 116, 166, 217, 267, 317, 367, 417, 467, 517, 566, 616]

RESULT_REGION_SKELETON = (472, 0, 100, 40)

CONST_PLAYERS_HASH = {"mario": 0, "luigi": 1, "peach": 2, "daisy": 3}

FRAME_WIDTH = 1280
FRAME_HEIGHT = 720

MIN_FRAME_TIME = 0.02

SAMPLING_TIME = 14.25

MAX_MILISECONDS_RUN = 10800000

POINTS_DICT = {
    1: 15,
    2: 12,
    3: 10,
    4: 9,
    5: 8, 
    6: 7,
    7: 6,
    8: 5,
    9: 4,
    10: 3,
    11: 2,
    12: 1
}

## Class Definitions

In [4]:
class Result:
    def __init__(self, netid: str, name: str, college: str, place: int, points: int, lap1: float, lap2: float, lap3: float):
        self.netid = netid
        self.name = name
        self.college = college

        self.college_id = getDocId(college)
        
        self.place = place
        self.points = points
        self.lap1 = lap1
        self.lap2 = lap2
        self.lap3 = lap3
        
        self.total_time = lap1 + lap2 + lap3

    @staticmethod
    def from_dict(source):
        return Result(
            netid = source["netid"],
            name = source["name"],
            college = source["college"],
            place = source["place"],
            points = source["points"],
            lap1 = source["lap1"],
            lap2 = source["lap2"],
            lap3 = source["lap3"]
        )

    def to_dict(self):
        return {
            "netid": self.netid,
            "name": self.name,
            "college": self.college,
            "college_id": self.college_id,
            "place": self.place,
            "points": self.points,
            "lap1": self.lap1,
            "lap2": self.lap2,
            "lap3": self.lap3,
            "total_time": self.total_time
        }

    def __repr__(self):
        return (f"---\n\n"
                f"NetId: {self.netid}\n"
                f"Name: {self.name}\n"
                f"College: {self.college} ({self.college_id})\n"
                f"Place: {self.place}\n"
                f"Points: {self.points}\n"
                f"Total Time: {self.total_time}\n"
                f"Splits: {self.lap1} | {self.lap2} | {self.lap3}\n\n")

class Race:
    def __init__(self, datetime: str, event: str, track: str, results: List[Result]):
        self.datetime = datetime
        self.event = event

        self.event_id = getDocId(event)

        self.race_number = -1

        self.race_id = getDocId(f"{event}-{str(race_number).zfill(3)}")
        
        self.track = track
        self.results = sorted(results, key = lambda result: result.total_time)

    @staticmethod
    def from_dict(source):
        results = [Result.from_dict(result) for result in source["results"]]
        return Race(
            datetime = source["datetime"],
            event = source["event"],
            race_number = source["race_number"],
            track = source["track"],
            results = results
        )

    def to_dict(self):
        results = [result.to_dict() for result in self.results]
        return {
            "datetime": self.datetime,
            "event": self.event,
            "event_id": self.event_id,
            "race_number": self.race_number,
            "race_id": self.race_id,
            "track": self.track,
            "results": results
        }

    def __repr__(self):
        return (f"Datetime: {self.datetime}\n"
                f"Event: {self.event} ({self.event_id})\n"
                f"Race Number: {self.race_number} ({self.race_id})\n"
                f"Track: {self.track}\n"
                f"Results:\n\n"
                f"{''.join([str(result) for result in self.results])}")

class BaseEvent:
    def __init__(self, name, location, description, start, end):
        self.name = name
        self.location = location
        self.description = description
        self.start = start
        self.end = end

    def __repr__(self):
        return (f"{self.name}:\n"
                f"{self.start} - {self.end}\n"
                f"{self.location}\n\n"
                f"{self.description}")



## Function Definitions

In [5]:
def getDocId(name: str):
    cleaned = re.sub(r"[/?*=#]", "", name)
    return cleaned.replace(" ", "_").lower()

# Consider multiprocessing instead of threading
def race_submission_worker():
    while True:
        data = write_queue.get()
        # write_queue.put(None) to shut down
        if data is None:
            break
        
        postRace(data)
        
        write_queue.task_done()

def findYalie(netid):
    headers = {
    	"Authorization": f"Bearer {YALIES_KEY}"
    }
    body = {
    	"filters": {"netid": netid}
    }
    request = requests.post("https://api.yalies.io/v2/people", headers=headers, json=body)
    return request.json()

def validateYalie(yalie):
    if (len(yalie) > 0):
        yalie_object = yalie[0]
    else:
        raise RuntimeError(f"Yalies response empty: {yalie}")
    
    if ("netid" in yalie_object) and ("first_name" in yalie_object) and ("last_name" in yalie_object) and ("year" in yalie_object) and ("college" in yalie_object):
        return {
            "netid": yalie_object["netid"],
            "first_name": yalie_object["first_name"],
            "last_name": yalie_object["last_name"],
            "year": yalie_object["year"],
            "college": yalie_object["college"]
        }
    else:
        raise RuntimeError(f"Yalies response incomplete: {yalie_object}")

def createNewUser(netid: str):
    yalies_data = validateYalie(findYalie(netid))
    college_id = getDocId(yalies_data["college"])
    user_data = {
        "netid": yalies_data["netid"],
        "first_name": yalies_data["first_name"],
        "last_name": yalies_data["last_name"],
        "year": yalies_data["year"],
        "college": yalies_data["college"],
        "points": 0,
        "race_count": 0,
        "races": []
    }

    return (db.collection("users").document(user_data["netid"]).set(user_data), yalies_data)

def updateUsers(race_object):
    for result in race_object.results:
        if not db.collection("users").document(result.netid).get().exists:
            createNewUser(result.netid)
        doc_ref = db.collection("users").document(result.netid)
        doc_ref.update({
            "points": firestore.Increment(result.points),
            "race_count": firestore.Increment(1),
            "races": firestore.ArrayUnion([{
                "race_id": race_object.race_id,
                "datetime": race_object.datetime,
                "event": race_object.event,
                "event_id": race_object.event_id,
                "track": race_object.track,
                "place": result.place,
                "points": result.points,
                "lap1": result.lap1,
                "lap2": result.lap2,
                "lap3": result.lap3,
                "total_time": result.total_time
            }])
        })

def initializeEvent(base_event: BaseEvent):
    event_data = {"name": base_event.name,
                    "event_id": getDocId(base_event.name),
                    "location": base_event.location,
                    "start": base_event.start,
                    "end": base_event.end,
                    "participants": [],
                    "races": [],
                    "colleges": [{"name": name,
                                    "college_id": getDocId(name),
                                    "points": 0,
                                    "race_count": 0} for name in COLLEGE_NAMES]}

    return db.collection("events").document(event_data["event_id"]).set(event_data)

# This can be more optimal, but the differentce between O(n) and O(n log n) here may not be that big or existent at all
def updateEvent(race_object):
    doc_ref = db.collection("events").document(race_object.event_id)
    doc = doc_ref.get()

    point_additions = {college: 0 for college in COLLEGE_NAMES} 
    race_additions = {college: 0 for college in COLLEGE_NAMES} 
    netids = []

    if doc.exists:
        data = doc.to_dict()
        colleges = data.get("colleges", [])
    else:
        raise RuntimeError(f"Event not found: {event_id}")
    
    for result in race_object.results:
        point_additions[result.college] += result.points
        race_additions[result.college] += 1
        netids.append(result.netid)

    for college in colleges:
        college["points"] += point_additions[college["name"]]
        college["race_count"] += race_additions[college["name"]]

    race = {
        "race_id": race_object.race_id,
        "datetime": race_object.datetime,
        "event": race_object.event,
        "event_id": race_object.event_id,
        "track": race_object.track,
        "winner_name": race_object.results[0].name,
        "winner_college": race_object.results[0].college,
        "winner_college_id": race_object.results[0].college_id,
        "winner_place": race_object.results[0].place,
        "winner_points": race_object.results[0].points
    }
        

    sorted_colleges = sorted(colleges, key = lambda college: college["points"], reverse = True)

    count = 0
    last_place = 0
    last_points = -1

    for college in sorted_colleges:
        count += 1
        
        if college["points"] != last_points:
            last_points = college["points"]
            last_place = count

        college["place"] = last_place

    return doc_ref.update({"participants": firestore.ArrayUnion(netids),
                            "colleges": sorted_colleges,
                            "races": firestore.ArrayUnion([race])})

def initializeCollege(name: str):
    college_data = {"name": name,
                    "college_id": getDocId(name),
                    "points": 0,
                    "races": 0,
                    "score": 0,
                    "students": []}

    return db.collection("colleges").document(college_data["college_id"]).set(college_data)

def initializeColleges(college_names: List[str]):
    responses = []
    
    for name in college_names:
        responses.append(initializeCollege(name))

    return responses

# This on the other hand is pretty non-optimal, can be brought to O(n) pretty easily during refinement
def updateColleges(race_object):
    responses = []

    for result in race_object.results:
        doc_ref = db.collection("colleges").document(result.college_id)
        doc = doc_ref.get()

        if doc.exists:
            data = doc.to_dict()
            students = data.get("students", [])
            points = data.get("points", 0)
            races = data.get("races", 0)
        else:
            raise RuntimeError(f"College not found: {result.college_id}")

        student_found = False
        
        for student in students:
            if student["netid"] == result.netid:
                student_found = True
                student["points"] += result.points
                student["races"] += 1
                break

        if not student_found:
            yalies_data = validateYalie(findYalie(result.netid))
            students.append({"netid": result.netid,
                             "name": result.name,
                             "year": yalies_data["year"],
                             "points": result.points,
                             "races": 1})

        sorted_students = sorted(students, key = lambda student: student["points"], reverse = True)
    
        count = 0
        last_place = 0
        last_points = -1
    
        for student in sorted_students:
            count += 1
            
            if student["points"] != last_points:
                last_points = student["points"]
                last_place = count
    
            student["place"] = last_place

            db.collection("users").document(student["netid"]).update({"college_rank": student["place"]})


        responses.append(doc_ref.update({"points": firestore.Increment(result.points),
                                         "races": firestore.Increment(1),
                                         "score": (points + result.points) / (races + 1),
                                         "students": sorted_students}))

    return responses

def postRace(race_object): 
    # Get and set race number
    event_doc = db.collection("events").document(getDocId(event_name)).get()
    event_data = event_doc.to_dict()
    race_number = len(event_data["races"]) + 1

    race_object.race_number = race_number
    
    # Add to races collection
    db.collection("races").document(race_object.race_id).set(race_object.to_dict())

    # Add to users collection
    updateUsers(race_object)

    # Add to events collection
    updateEvent(race_object)

    #Add to colleges collection
    updateColleges(race_object)

## Main Loop

In [ ]:
# Tesseract inconsistency
# Better error logging

"""
States explained:
State 0: Pre-Race (Login)
State 1: Pre-Race (Menus)
State 2: White Loading Screen
State 3: Title Animation (Scrape title)
State 4: Pre-Race (Waiting for Go!)
State 5: Race (Scraping laps)
State 6: Race Results (Scrape results)
"""

# Outside loop

# Later, make this a class
# Initialize the players
players = [{'lap_region': LAP_REGIONS[i], 'finish_region': FINISH_REGIONS[i], 'lap': 0, 'transition_counter': 0, 'netid': None} for i in range(4)]

# Set up the video capture streaming
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)

# Initialize global variables
checkpoint = 0
state = 0
lap_name_sampling_start = 0
normal_texts = {}
gray_texts = {}
track_name = ''
results = [[] for i in range(4)]
places = [0] * 4
problem_names = []
elapsed = 0
players_hash = CONST_PLAYERS_HASH
write_queue = queue.Queue()
player_index = 0
race_time = ""

# TEMPORARY Hardcoded Values
event = BaseEvent(
    name="Trumbull Grand Prix",
    location="Trumbull Dining Hall",
    description="Spice up meatloaf night with some Mario Kart with friends!",
    start="2025-06-15T12:00:00",
    end="2025-06-15T17:00:00"
)

# Set up Firestore
env_path = Path('..')/'.env'
load_dotenv(dotenv_path=env_path)

if not firebase_admin._apps:
    cred = credentials.Certificate(os.environ.get("GOOGLE_APPLICATION_CREDENTIALS"))
    firebase_admin.initialize_app(cred)

db = firestore.client()

# Run firebase worker on a separate thread
threading.Thread(target=race_submission_worker, daemon=True).start()

# Initialize event if it doesn't exist

# Run gameplay loop
for _ in range(MAX_MILISECONDS_RUN):
    # Get the data for the current frame
    ret, frame = cap.read()
    if not ret:
        print("Frame capture failed")
        break
    
    # Turn frame into an image and display
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(frame_rgb)
    draw = ImageDraw.Draw(pil_img)
    
    match state:
        # Pre-Race (Login) -> Transition when all four players have signed up
        # Action: Fetch data from Yalies as needed
        case 0:
            netid = input("(Simulate scan) Enter netid: ")

            try:
                if not db.collection("users").document(netid).get().exists:
                    result, data = createNewUser(netid)

                players[player_index]['netid'] = netid
                players[player_index]['name'] = data['first_name'] + ' ' + data['last_name']
                players[player_index]['year'] = data['year']
                players[player_index]['college'] = data['college']
                player_index += 1
            except Exception as error:
                print(f'Error logging in user with netid "{netid}": {error}')
            
            if not any([p['netid'] is None for p in players]):
                # State transition
                state = 1
        
        # Pre-Race (Menus) -> Transition when screen is white
        # Action: Click "A" repeatedly
        case 1:
            (x, y, w, h) = LOADING_REGION
            
            region = frame[y:y+h, x:x+w]
            gray_region = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY)
            thresh = cv2.threshold(gray_region, 240, 255, cv2.THRESH_BINARY)[1]
            
            draw.rectangle([x, y, x + w, y + h], outline="red", width=3)
    
            if sum(sum(thresh == 255)) / len(thresh) / len(thresh[0]) > 0.95:
                # State transition
                state = 2
    
            # Action placeholder
            print("Click A")
    
        # White Loading Screen -> Transition when screen is not white
        # Action: None
        case 2:
            (x, y, w, h) = LOADING_REGION
            
            region = frame[y:y+h, x:x+w]
            gray_region = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY)
            thresh = cv2.threshold(gray_region, 240, 255, cv2.THRESH_BINARY)[1]
            
            draw.rectangle([x, y, x + w, y + h], outline="red", width=3)
    
            if sum(sum(thresh == 255)) / len(thresh) / len(thresh[0]) < 0.95:
                lap_name_sampling_start = time.time()
                normal_texts = {}
                gray_texts = {}

                # Set the time of the race
                race_time = datetime.now().isoformat()
    
                # State transition
                state = 3
    
        # Title Animation -> Transition after fixed time
        # Action: Tesseract OCR and store matching track name
        case 3:        
            (x, y, w, h) = TRACK_NAME_REGION
            
            region = frame[y:y+h, x:x+w]
            region_image = Image.fromarray(region)
            gray_region = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY)
            gray_image = Image.fromarray(gray_region)
            
            draw.rectangle([x, y, x + w, y + h], outline="red", width=3)
            
            text_normal = pytesseract.image_to_string(region_image, lang='eng').lower().strip()
            text_gray = pytesseract.image_to_string(gray_image, lang='eng').lower().strip()
    
            if text_normal in normal_texts:
                normal_texts[text_normal] += 1
            elif len(text_normal) > 0:
                normal_texts[text_normal] = 1
    
            if text_gray in gray_texts:
                gray_texts[text_gray] += 1
            elif len(text_gray) > 0:
                gray_texts[text_gray] = 1
    
            if time.time() - sampling_start > SAMPLING_TIME:
                normal_text = max(normal_texts, key=normal_texts.get)
                gray_text = max(gray_texts, key=gray_texts.get)
    
                if normal_text in TRACK_TRANSLATIONS:
                    track_name = TRACK_TRANSLATIONS[normal_text]
                elif gray_text in TRACK_TRANSLATIONS:
                    track_name = TRACK_TRANSLATIONS[gray_text]
                else:
                    track_name = f'UNKNOWN: {normal_text}'
                
                # State transition
                state = 4
    
        # Pre-Race -> Transition when "Go!" appears
        # Action: None
        case 4:
            (x, y, w, h) = GO_REGION
    
            region = frame[y:y+h, x:x+w]
            gray_region = cv2.cvtColor(gray_region, cv2.COLOR_BGR2GRAY)
            
            draw.rectangle([x, y, x + w, y + h], outline="red", width=3)
    
            match_go = sum(sum(np.abs(gray_region - GO_MASK) < 5)) / (len(GO_MASK) * len(GO_MASK[0]))
            
            if (match_go > 0.15):
                # Initialize variables for race start
                checkpoint = time.time()
    
                # State transition
                state = 5
    
        # Race -> Transition when all players hit "Finish" and wait fixed time
        # Action: Store lap times during race
        case 5:
            for index, player in enumerate(players):
                (x, y, w, h) = player['lap_region']
                
                region = frame[y:y+h, x:x+w]
                gray_region = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY)
                thresh = cv2.threshold(gray_region, 220, 255, cv2.THRESH_BINARY)[1]
        
                draw.rectangle([x, y, x + w, y + h], outline="red", width=3)
                
                match_1 = sum(sum(thresh == LAP_MASKS[index][0])) / (len(thresh) * len(thresh[0]))
                match_2 = sum(sum(thresh == LAP_MASKS[index][1])) / (len(thresh) * len(thresh[0]))
                match_3 = sum(sum(thresh == LAP_MASKS[index][2])) / (len(thresh) * len(thresh[0]))
    
                # Advance Laps
                if (player['lap'] == 1):
                    if (match_2 > match_1):
                        if (player['transition_counter'] >= 50):
                            player['lap'] = 2
                            player['transition_counter'] = 0
                            results[index].append(time.time() - checkpoint - 2.5)
                        else:
                            player['transition_counter'] += 1
                    else:
                        player['transition_counter'] = 0
                if (player['lap'] == 2):
                    if (match_3 > match_2):
                        if (player['transition_counter'] >= 50):
                            player['lap'] = 3
                            player['transition_counter'] = 0
                            results[index].append(time.time() - checkpoint - 2.5)
                        else:
                            player['transition_counter'] += 1
                    else:
                        player['transition_counter'] = 0
        
                # Find "Finish"
                if player['lap'] < 4:
                    (x, y, w, h) = player['finish_region']
            
                    region = frame[y:y+h, x:x+w]
                    gray_region = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY)
                    
                    draw.rectangle([x, y, x + w, y + h], outline='red', width=3)
                    
                    match_finish = sum(sum(np.abs(gray_region - FINISH_REGIONS[index]) < 5)) / (len(gray_region) * len(gray_region[0]))    
                    
                    if (match_finish > 0.18):
                        results[index].append(time.time() - checkpoint - 1)
                        player["lap"] = 4
    
            if all([p['lap'] == 4 for p in players]):
                # Wait for results screen to appear
                time.sleep(2.65)
                
                # State transition
                state = 6
    
        # Race results -> Transition after fixed time
        # Action: Scrape places/results from race, submit database job
        case 6:
            places = [0] * 4
            problem_names = []
            players_hash = CONST_PLAYERS_HASH
            
            for place_0, y_value in enumerate(RESULT_REGION_Y_VALUES):
                place = place_0 + 1
    
                (x, y, w, h) = RESULT_REGION_SKELETON
                y = y_value
    
                region = frame[y:y+h, x:x+w]
                gray_region = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY)
                thresh = cv2.threshold(gray_region, 135, 255, cv2.THRESH_BINARY_INV)[1]
                gray_image = Image.fromarray(gray_region)
                thresh_image = Image.fromarray(thresh)
                
                draw.rectangle([x, y, x + w, y + h], outline="red", width=3)
                
                gray_text = pytesseract.image_to_string(gray_image, lang='eng').lower().strip()
                thresh_text = pytesseract.image_to_string(thresh_image, lang='eng').lower().strip()
                
                if gray_text in players_hash:
                    index = players_hash[gray_text]
            
                    if index >= 0:
                        places[index] = place
                        players_hash[gray_text] = -1
                    else:
                        print("Problem: duplicate player in places")
                elif thresh_text in PLAYERS_HASH:
                    index = players_hash[thresh_text]
            
                    if index >= 0:
                        places[index] = place
                        players_hash[thresh_text] = -1
                    else:
                        print("Problem: duplicate player in places")
                else:
                    problem_names.extend([gray_text, thresh_text])
    
            if any(map(lambda place: place == 0, places)):
                print("Problem: not all places assigned")
                print("\n".join(problem_names))
            
            results_objects = [
                Result(
                    netid = player['netid'],
                    name = player['name'],
                    college = player['college'],
                    place = places[index],
                    points = POINTS_DICT[places[index]],
                    lap1 = results[index][0],
                    lap2 = results[index][1],
                    lap3 = results[index][2]
                ) for index, player in players
            ]

            # Race number must be determined in the thread
            race_object = Race(
                datetime = race_time,
                event = event.name,
                track = track_name,
                results = results_objects
            )

            write_queue.put(race_object)

            # Reset values
            for player in players:
                player["lap"] = 1
                player['transition_counter'] = 0
                player['netid'] = None
            results = [[] for i in range(4)]
            
            # State transition
            state = 6
    
    # Display frame
    display(pil_img)
    
    # Wait (don't unnecessarily compute)
    elapsed = time.time() - start
    while(elapsed < MIN_FRAME_TIME):
        elapsed = time.time() - start
    
    # Clear output for next cycle
    clear_output(wait=True)

# End the worker
write_queue.put(None)

## Utility Code: Initialize Events and Colleges

In [7]:
COLLEGE_NAMES = ["Benjamin Franklin", "Berkeley", "Branford", "Davenport", "Ezra Stiles", "Grace Hopper", "Jonathan Edwards", "Morse", "Pauli Murray", "Pierson", "Saybrook", "Silliman", "Timothy Dwight", "Trumbull"]

# Function definitions
def initializeCollege(name: str):
    college_data = {"name": name,
                    "college_id": getDocId(name),
                    "points": 0,
                    "races": 0,
                    "score": 0,
                    "students": []}

    return db.collection("colleges").document(college_data["college_id"]).set(college_data)

def initializeColleges(college_names: List[str]):
    responses = []
    
    for name in college_names:
        responses.append(initializeCollege(name))

    return responses

def initializeEvent(base_event: BaseEvent):
    event_data = {"name": base_event.name,
                    "event_id": getDocId(base_event.name),
                    "location": base_event.location,
                    "start": base_event.start,
                    "end": base_event.end,
                    "participants": [],
                    "races": [],
                    "colleges": [{"name": name,
                                    "college_id": getDocId(name),
                                    "points": 0,
                                    "race_count": 0} for name in COLLEGE_NAMES]}

    return db.collection("events").document(event_data["event_id"]).set(event_data)

In [ ]:
# Initialize colleges
# initializeColleges(COLLEGE_NAMES)

# Initialize event
# initializeEvent(__event__)